Это ноутбук для обучения YOLOv8 для infraPPE-monitor  
https://github.com/ferrovovan/infraPPE-monitor

# Шаг 0: Подготовка среды и установка библиотек

In [ ]:
!pip install kaggle
!pip install ultralytics


# Вместо файла kaggle.json мы будем использовать переменные окружения.
import os
os.environ['KAGGLE_USERNAME'] = 'vash_username' # <-- ЗАМЕНИТЕ НА ВАШЕ ИМЯ ПОЛЬЗОВАТЕЛЯ KAGGLE!
os.environ['KAGGLE_KEY'] = input("Введите KAGGLE_KEY: ")


import ultralytics
ultralytics.checks()

# Шаг 1: Скачивание датасета


In [2]:

#DATASET_REPO = 'andrewmvd/hard-hat-detection'
#DATASET_TYPE = 'Pascal VOC'
DATASET_REPO = 'shlokraval/ppe-dataset-yolov8'
DATASET_TYPE = 'YOLO TXT'

repo_name = DATASET_REPO.split('/')[-1]
DATASET_ARCHIVE_NAME = f'{repo_name}.zip'
DATASET_DIR = 'dataset'


!kaggle datasets download -d {DATASET_REPO} #-O {DATASET_ARCHIVE_NAME}

# 4. Распакуйте архив в отдельную папку для удобства
!mkdir -p {DATASET_DIR}
!unzip -q {DATASET_ARCHIVE_NAME} -d {DATASET_DIR}

# Проверяем, что внутри (полезно для отладки)
print(f"Содержимое папки {DATASET_DIR}:")
!ls {DATASET_DIR}

Dataset URL: https://www.kaggle.com/datasets/shlokraval/ppe-dataset-yolov8
License(s): apache-2.0
 99% 2.32G/2.35G [00:29<00:00, 232MB/s]
100% 2.35G/2.35G [00:29<00:00, 86.6MB/s]
Содержимое папки dataset:
data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid


# Шаг 2: Подготовка датасета


In [ ]:
import glob
import os
import random
import shutil
import xml.etree.ElementTree as ET
import yaml # Для чтения data.yaml в случае YOLO TXT

# Глобально определяем CONFIG_PATH и CLASSES, которые будут использоваться далее
CONFIG_PATH = None
CLASSES = []

In [ ]:
if DATASET_TYPE == 'Pascal VOC':
    print("Обнаружен тип датасета: Pascal VOC. Выполняем конвертацию и подготовку.")

    # --- 2.1 Конвертация из XML (Pascal VOC) в TXT (YOLO) ---

    def convert_xml_to_yolo(xml_file, classes):
        """Конвертирует один XML файл в формат YOLO (строка)"""
        tree = ET.parse(xml_file)
        root = tree.getroot()
        size = root.find('size')
        if size is None:
            print(f"Warning: 'size' элемент не найден в XML файле {xml_file}. Пропускаем конвертацию.")
            return []

        w = int(size.find('width').text)
        h = int(size.find('height').text)

        yolo_annotations = []
        for obj in root.findall('object'):
            name = obj.find('name').text
            if name not in classes:
                continue
            cls_id = classes.index(name)
            bndbox = obj.find('bndbox')
            xmin = int(bndbox.find('xmin').text)
            ymin = int(bndbox.find('ymin').text)
            xmax = int(bndbox.find('xmax').text)
            ymax = int(bndbox.find('ymax').text)

            # Конвертация абсолютных координат в относительные (YOLO формат)
            x_center = (xmin + xmax) / 2.0
            y_center = (ymin + ymax) / 2.0
            width = xmax - xmin
            height = ymax - ymin

            x_center /= w
            y_center /= h
            width /= w
            height /= h

            yolo_annotations.append(f"{cls_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

        return yolo_annotations

    # Определяем классы для Pascal VOC
    CLASSES = ['helmet', 'head', 'person']  # Эти классы будут использоваться для преобразованного датасета
    INPUT_IMAGES_DIR = 'dataset/images'  # Предполагается, что изображения Pascal VOC находятся здесь
    INPUT_ANNOTATIONS_DIR = 'dataset/annotations'  # Предполагается, что XML Pascal VOC находятся здесь
    OUTPUT_BASE_DIR = 'ppe_yolo_data'  # Новая директория для преобразованных данных YOLO
    OUTPUT_LABELS_DIR = os.path.join(OUTPUT_BASE_DIR, 'labels')
    OUTPUT_IMAGES_DIR_BASE = os.path.join(OUTPUT_BASE_DIR, 'images')

    os.makedirs(OUTPUT_LABELS_DIR, exist_ok=True)
    os.makedirs(OUTPUT_IMAGES_DIR_BASE, exist_ok=True)

    xml_files = glob.glob(os.path.join(INPUT_ANNOTATIONS_DIR, '*.xml'))
    converted_label_count = 0
    if xml_files:
        print(f"Начало конвертации {len(xml_files)} XML файлов в формат YOLO TXT...")
        for xml_file in xml_files:
            try:
                yolo_data = convert_xml_to_yolo(xml_file, CLASSES)
                if yolo_data: # Проверяем, есть ли данные для записи
                    img_name_base = os.path.basename(xml_file).replace('.xml', '')
                    output_txt_path = os.path.join(OUTPUT_LABELS_DIR, img_name_base + '.txt')
                    with open(output_txt_path, 'w') as f:
                        f.write('\n'.join(yolo_data))
                    converted_label_count += 1
            except Exception as e:
                print(f"Ошибка при конвертации {xml_file}: {e}")

        print(f"Конвертировано аннотаций в TXT: {converted_label_count}")
    else:
        print(f"Внимание: Не найдено XML файлов в {INPUT_ANNOTATIONS_DIR}. Проверьте путь или тип датасета.")


    print("\n--- 2.2 Разделение на train/val и копирование файлов ---")

    # Создаем финальную структуру папок для разделенного датасета
    os.makedirs(os.path.join(OUTPUT_IMAGES_DIR_BASE, 'train'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_IMAGES_DIR_BASE, 'val'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_LABELS_DIR, 'train'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_LABELS_DIR, 'val'), exist_ok=True)

    # Получаем список всех изображений, для которых есть конвертированные аннотации
    image_files_with_labels = []
    for label_file in glob.glob(os.path.join(OUTPUT_LABELS_DIR, '*.txt')):
        img_basename = os.path.basename(label_file).replace('.txt', '')
        # Проверяем на наличие общих расширений изображений
        found_image = False
        for ext in ['.png', '.jpg', '.jpeg']:
            img_path = os.path.join(INPUT_IMAGES_DIR, img_basename + ext)
            if os.path.exists(img_path):
                image_files_with_labels.append(img_path)
                found_image = True
                break
        if not found_image:
            print(f"Warning: Изображение для аннотации {img_basename}.txt не найдено в {INPUT_IMAGES_DIR} с расширениями .png, .jpg, .jpeg.")

    random.shuffle(image_files_with_labels)

    # Определяем пропорции для разделения (например, 80% train, 20% val)
    train_split = int(len(image_files_with_labels) * 0.8)
    train_images = image_files_with_labels[:train_split]
    val_images = image_files_with_labels[train_split:]

    def copy_image_and_label_files(image_list, image_dest, label_dest):
        for img_path in image_list:
            img_name = os.path.basename(img_path)
            img_basename_no_ext = os.path.splitext(img_name)[0]
            label_name = img_basename_no_ext + '.txt'

            shutil.copy(img_path, os.path.join(image_dest, img_name))
            src_label_path = os.path.join(OUTPUT_LABELS_DIR, label_name)
            if os.path.exists(src_label_path):
                shutil.copy(src_label_path, os.path.join(label_dest, label_name))
            else:
                print(f"Warning: Файл разметки {label_name} не найден для изображения {img_name}. Пропускаем.")

    copy_image_and_label_files(train_images, os.path.join(OUTPUT_IMAGES_DIR_BASE, 'train'), os.path.join(OUTPUT_LABELS_DIR, 'train'))
    copy_image_and_label_files(val_images, os.path.join(OUTPUT_IMAGES_DIR_BASE, 'val'), os.path.join(OUTPUT_LABELS_DIR, 'val'))

    print(f"Train images: {len(train_images)}, Validation images: {len(val_images)}")

    print("\n--- 2.3 Создание файла data.yaml ---")

    yaml_content = f"""
train: {OUTPUT_BASE_DIR}/images/train
val: {OUTPUT_BASE_DIR}/images/val

nc: {len(CLASSES)}
names: {CLASSES}
"""
    CONFIG_PATH = 'ppe_data.yaml'  # Устанавливаем CONFIG_PATH для случая Pascal VOC
    with open(CONFIG_PATH, 'w') as f:
        f.write(yaml_content)

    print(f"Файл {CONFIG_PATH} успешно создан.")
    !cat {CONFIG_PATH}

In [ ]:

if DATASET_TYPE == 'YOLO TXT':
    print("Обнаружен тип датасета: YOLO TXT. Используем существующие файлы датасета.")
    # В этом случае датасет предполагается уже загруженным и структурированным.
    # Файл `data.yaml` ожидается в каталоге 'dataset' загруженного датасета.
    CONFIG_PATH = os.path.join(DATASET_DIR, 'data.yaml')  # Устанавливаем CONFIG_PATH для случая YOLO TXT

    if os.path.exists(CONFIG_PATH):
        print(f"Используем файл конфигурации датасета: {CONFIG_PATH}")
        # Пытаемся прочитать классы из существующего data.yaml для дальнейшего использования, если это необходимо
        try:
            with open(CONFIG_PATH, 'r') as f:
                data_config = yaml.safe_load(f)
                if 'names' in data_config:
                    CLASSES = data_config['names']
                    print(f"Классы, обнаруженные в {CONFIG_PATH}: {CLASSES}")
                else:
                    print(f"Предупреждение: 'names' не найдены в {CONFIG_PATH}. Переменная CLASSES будет пустой.")
        except Exception as e:
            print(f"Ошибка при чтении {CONFIG_PATH}: {e}. Переменная CLASSES будет пустой.")
    else:
        print(f"Ошибка: Файл {CONFIG_PATH} не найден для типа датасета YOLO TXT. Пожалуйста, убедитесь, что он существует.")
        # Если CONFIG_PATH не найден, последующие шаги, зависящие от него, могут завершиться неудачей.
        CONFIG_PATH = None
        CLASSES = []  # Убедимся, что CLASSES сброшен, если data.yaml не найден

else:
    print(f"Неизвестный тип датасета: {DATASET_TYPE}. Невозможно подготовить данные.")
    CONFIG_PATH = None
    CLASSES = []


In [ ]:
if CONFIG_PATH:
    print(f"\nФинальный путь к файлу конфигурации датасета (CONFIG_PATH) установлен в: {CONFIG_PATH}")
else:
    print("\nФинальный путь к файлу конфигурации датасета (CONFIG_PATH) не был установлен корректно.")

# Шаг 3: Обучение модели


In [3]:
from ultralytics import YOLO
from torch.cuda import is_available as is_gpu
import os
from google.colab import drive

drive.mount('/content/drive')

# Путь к скачанному файлу data.yaml внутри папки dataset
CONFIG_PATH = 'dataset/data.yaml'
GDRIVE_PATH = '/content/drive/MyDrive/'

# Определяем параметры обучения
TOTAL_EPOCHS = 20
SAVE_PERIOD = 3  # сохраняет доп. файлы каждые 3 эпох
IMG_SIZE = 640
if is_gpu:
    BATCH_SIZE = 96  # Для Tesla T4, 15095MiB
else:
    print("На CPU слишком долго. Борода.")
    quit()
    #BATCH_SIZE = -1

PROJECT_DIR = GDRIVE_PATH + 'train_ppe_v1' # Место сохранения результатов в Colab
EXPERIMENT_NAME = 'yolov8_ppe_nano'
START_WEIGHTS_NAME = 'yolov8n.pt'
# START_WEIGHTS_NAME = 'best.pt'


# Инициализации весов ---
best_weights_path = os.path.join(PROJECT_DIR, EXPERIMENT_NAME, 'weights', 'last.pt')
# print(best_weights_path)

if os.path.exists(best_weights_path):
    current_weights = best_weights_path
    print(f"Обнаружены предыдущие лучшие веса: {current_weights}.\n Обучение будет продолжено с них.")
else:
    current_weights = START_WEIGHTS_NAME
    print(f"Предыдущие лучшие веса не найдены.\n Обучение начнется с предобученной модели: {current_weights}")

# Создаем директорию проекта, если ее нет
os.makedirs(PROJECT_DIR, exist_ok=True)


# 1. Загружаем модель с последними сохраненными весами
model = YOLO(current_weights)

# 2. Запускаем обучение на EPOCHS_PER_CYCLE эпох
# Каждому циклу присваиваем уникальное имя, чтобы результаты сохранялись отдельно
results = model.train(
    data=CONFIG_PATH,
    epochs=TOTAL_EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    project=PROJECT_DIR,
    name=f'{EXPERIMENT_NAME}',
    save=True,             # Сохранять модель (weights/best.pt и weights/last.pt)
    resume=True           # Возобновляет обучение с того места, где оно было прервано
    #resume=False           # Для нового обучения
)

print("\n--- Обучение завершено! ---")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Обнаружены предыдущие лучшие веса: /content/drive/MyDrive/train_ppe_v1/yolov8_ppe_nano/weights/best.pt.
 Обучение будет продолжено с них.
Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=96, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, ma

KeyboardInterrupt: 

# Шаг 4. Скопировать пути на диск

In [ ]:
!cp {PROJECT_DIR}/{EXPERIMENT_NAME}/weights/best.pt  {PROJECT_DIR}/{EXPERIMENT_NAME}/detect_ppe_nano.pt

In [ ]:
# Для переноса (сохранения) весов из локального на диск.
import shutil
from pathlib import Path

local_dir = Path('/content/train_ppe_v1/yolov8_ppe_nano/weights')
gdrive_dir = Path('/content/gdrive/MyDrive/train_ppe_v1/yolov8_ppe_nano/weights')
if os.path.exists(local_dir):
    gdrive_dir.mkdir(parents=True, exist_ok=True)
    for p in local_dir.glob('*.pt'):
        shutil.copy2(p, gdrive_dir / p.name)
    print("Копирование завершено:", list(gdrive_dir.glob('*')))